# Chatbot vs human agent tickets: Mann-Whitney U test

**Mann-Whitney U Test** · a worked example from the StatNex teaching guide

**The story.** An online shop answers support tickets two ways, and recorded the minutes needed to resolve 27 tickets: 18 handled by a chatbot and 9 by a human agent. The chatbot is quick for most customers, but four stuck tickets took 30 to 70 minutes.

**The question.** If we pick one chatbot ticket and one human ticket at random, which is more likely to be solved first?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/mann_whitney_u.csv` (18 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Chatbot,Human agent
5.0,14.0
30.0,12.0
4.0,16.0
7.0,13.0
6.0,19.0
42.0,11.0
3.0,15.0
5.0,17.0
9.0,13.0
6.0,
55.0,
4.0,
8.0,
7.0,
6.0,
5.0,
9.0,
70.0,

These cells save `mann_whitney.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile mann_whitney.py
"""The Mann-Whitney critical-value table, computed rather than looked up.

A printed table covers only the sample sizes somebody chose to publish; this covers
whatever sizes your data have. SciPy computes exact P values for a data set, but not
the null distribution itself, which is what a critical value is read from.
"""

import math

import numpy as np
from scipy import stats

#: Past this group size printed tables stop, and the normal approximation stands in.
EXACT_LIMIT = 50


def rank_sum_null(n1, n2):
    """P(U = u) for u = 0 .. n1*n2 when there are no ties.

    Every choice of which n1 of the n1 + n2 ranks belong to the first group is equally
    likely, so the distribution is a count of subsets of 1..N with n1 members, by their
    sum -- built one rank at a time. U is the rank sum less its smallest value.
    """
    total = n1 + n2
    top = n1 * total - n1 * (n1 - 1) // 2  # the largest possible rank sum
    ways = np.zeros((n1 + 1, top + 1))  # ways[k, w]: k ranks chosen so far, summing to w
    ways[0, 0] = 1.0
    for rank in range(1, total + 1):
        for k in range(min(rank, n1), 0, -1):
            ways[k, rank:] += ways[k - 1, :-rank]
    lowest = n1 * (n1 + 1) // 2
    null = ways[n1, lowest:lowest + n1 * n2 + 1]
    return null / null.sum()


def critical_u(n1, n2, alpha):
    """The largest U the table calls significant at tail probability ``alpha``.

    -1 when even U = 0 is not small enough: then nothing is significant.
    """
    if max(n1, n2) > EXACT_LIMIT:
        mean = n1 * n2 / 2
        sd = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
        return float(np.floor(mean + stats.norm.ppf(alpha) * sd))
    eligible = np.flatnonzero(np.cumsum(rank_sum_null(n1, n2)) <= alpha)
    return float(eligible[-1]) if eligible.size else -1.0


def hodges_lehmann_interval(a, b, level=0.95):
    """The interval for the shift of a over b that R's wilcox.test(a, b, conf.int=TRUE) gives.

    R 4.6's rule. With both groups under 50 the interval is exact. Without ties it is the
    q-th smallest and q-th largest difference a_i - b_j, q the (1 - level)/2 point of U.
    With ties, the limits are the differences at which the exact tail probability of the
    rank sum -- counted over every split of the ranks actually observed at that shift,
    ties included -- first exceeds (1 - level)/2. Either way it covers at least the level.
    From 50 a group on, it is where the tie-corrected, continuity-corrected z crosses
    +-z: the statistic only changes at a difference, so each limit is one.

    Returns (lower, upper), both NaN when no interval reaches the level (R: -Inf to Inf).
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    total = n1 + n2
    diffs = np.sort(np.subtract.outer(a, b).ravel())
    alpha = 1 - level
    toler = 10 * np.finfo(float).eps  # R's allowance for a tail equal to alpha/2
    if max(n1, n2) < EXACT_LIMIT and len(np.unique(np.concatenate([a, b]))) == total:
        null = rank_sum_null(n1, n2)
        cumulative = np.cumsum(null)
        q = int(np.argmax(cumulative >= alpha / 2 - toler))  # R's qwilcox
        if cumulative[q] <= alpha / 2 + toler:
            q += 1
        if q == 0:
            return np.nan, np.nan
        return diffs[q - 1], diffs[len(diffs) - q]
    if max(n1, n2) < EXACT_LIMIT:
        tables = {}

        def at_most(doubled, size, cut):
            # P(sum of `size` of the doubled ranks drawn at random <= cut), every split
            # equally likely: a count of subsets by size and sum.
            key = (tuple(sorted(doubled)), size)
            if key not in tables or len(tables[key]) <= cut:
                top = max(cut, size * (total + 1))
                ways = np.zeros((size + 1, top + 1))
                ways[0, 0] = 1.0
                for rank in doubled:
                    if rank <= top:
                        ways[1:, rank:] += ways[:-1, :top + 1 - rank].copy()
                tables[key] = ways[size] / math.comb(len(doubled), size)  # of every split
            return min(tables[key][:cut + 1].sum(), 1.0) if cut >= 0 else 0.0

        def tail(mu, upper):
            # The exact tail of the rank sum of a - mu against b, given its ties: the
            # lower tail P(W <= w), or the upper P(W >= w) as the second group's lower tail.
            doubled = np.rint(2 * stats.rankdata(np.concatenate([a - mu, b]))).astype(int)
            observed = int(doubled[:n1].sum())
            if upper:
                return at_most(doubled, n2, total * (total + 1) - observed)
            return at_most(doubled, n1, observed)

        return _scan(diffs, tail, alpha / 2 + toler)
    _, ties_a = np.unique(a, return_counts=True)
    _, ties_b = np.unique(b, return_counts=True)
    tie_sum = (ties_a**3 - ties_a).sum() + (ties_b**3 - ties_b).sum()
    sd = np.sqrt(n1 * n2 / 12 * ((total + 1) - tie_sum / (total * (total - 1))))
    values, counts = np.unique(diffs, return_counts=True)
    deviation = len(diffs) - np.cumsum(counts) - n1 * n2 / 2  # W just above each, less E(W)
    z_steps = (deviation - np.sign(deviation) * 0.5) / sd
    crit = stats.norm.ppf(1 - alpha / 2)
    return tuple(values[np.flatnonzero(z_steps <= c)[0]] if (z_steps <= c).any() else values[-1]
                 for c in (crit, -crit))


def _scan(sorted_values, tail, limit):
    """R 4.6's exact interval with ties: walk in from each end until the tail passes limit.

    The lower limit is the first value past which the upper tail, taken midway to the
    next value, exceeds ``limit``; the upper limit the same from the top with the lower
    tail. NaN for both when even beyond the outermost value the tail is that large.
    """
    count = len(sorted_values)
    if tail(sorted_values[0] - 1, True) > limit or tail(sorted_values[-1] + 1, False) > limit:
        return np.nan, np.nan
    lower = sorted_values[-1]
    for k in range(count - 1):
        if tail((sorted_values[k] + sorted_values[k + 1]) / 2, True) > limit:
            lower = sorted_values[k]
            break
    upper = sorted_values[0]
    for k in range(count - 2, -1, -1):
        if tail((sorted_values[k] + sorted_values[k + 1]) / 2, False) > limit:
            upper = sorted_values[k + 1]
            break
    return lower, upper

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from mann_whitney import critical_u, hodges_lehmann_interval

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per group. Text and blank cells are skipped, so the groups may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
first, second = "Chatbot", "Human agent"
a = pd.to_numeric(data[first], errors="coerce").dropna().to_numpy(float)
b = pd.to_numeric(data[second], errors="coerce").dropna().to_numpy(float)

## Descriptive statistics

In [ ]:
# Both groups are ranked together; tied values share the average of their ranks.
# U1 counts how often a value of the second group beats one of the first.
n1, n2 = len(a), len(b)
ranks = stats.rankdata(np.concatenate([a, b]))
rank_a, rank_b = ranks[:n1], ranks[n1:]
u1 = n1 * n2 + n1 * (n1 + 1) / 2 - rank_a.sum()
u2 = n1 * n2 - u1
descriptive = pd.DataFrame({"n": [n1, n2], "mean rank": [rank_a.mean(), rank_b.mean()],
                            "U": [u1, u2]}, index=[first, second])
print(descriptive)

## Test by critical-value table

In [ ]:
# The statistic is the smaller U. The table assumes no ties, so with ties it cannot be
# used and the normal approximation below is the test to read.
u = min(u1, u2)
tied = len(np.unique(ranks)) < n1 + n2
if tied:
    verdict = "ties present: the table does not apply (see the normal approximation)"
elif u <= critical_u(n1, n2, 0.01 / 2):
    verdict = "significant at 1% (two-sided)"
elif u <= critical_u(n1, n2, 0.05 / 2):
    verdict = "significant at 5% (two-sided)"
else:
    verdict = "not significant at 5% (two-sided)"
print(f"\nU = {u}: {verdict}")

## Normal-approximation test

In [ ]:
# If the two groups came from the same population, U would sit near E(U) = n1 n2 / 2.
# z says how many standard deviations it is away, and P is read from the normal curve.
# The variance carries the tie correction sum(t^3 - t) over groups of tied values.
# scipy.stats.mannwhitneyu(method="asymptotic") gives P with that same variance; z is
# read back from P. The continuity correction takes 0.5 off |U - E(U)|, because a
# count is being read off a smooth curve; z never goes below 0.
total = n1 + n2
_, tie_sizes = np.unique(ranks, return_counts=True)
expected = n1 * n2 / 2
variance = n1 * n2 / 12 * ((total + 1) - (tie_sizes**3 - tie_sizes).sum() / (total * (total - 1)))
rows = {}
plain = stats.mannwhitneyu(a, b, method="asymptotic", use_continuity=False)
rows["no correction"] = [stats.norm.isf(plain.pvalue / 2), plain.pvalue]
corrected = stats.mannwhitneyu(a, b, method="asymptotic", use_continuity=True)
rows["continuity correction"] = [stats.norm.isf(corrected.pvalue / 2), corrected.pvalue]
normal = pd.DataFrame.from_dict(rows, orient="index", columns=["z", "P (two-sided)"])
print(f"\nE(U) = {expected}, V(U) = {variance:.6g}")
print(normal)

## Exact test

In [ ]:
# Under the null hypothesis every split of the N midranks into groups of n1 and n2 is
# equally likely, so counting the splits by their rank sum gives the exact distribution,
# ties included. Doubled midranks are whole numbers, so the count is an array indexed by
# the doubled sum, built one value at a time over the smaller group's size. P is twice the
# smaller tail, capped at 1, as R 4.6's wilcox.test(exact = TRUE). Like the app, the count
# is skipped past 50 million cells of work, where only the normal approximation is shown.
# Written out because scipy's method="exact" assumes there are no ties.
doubled = np.rint(2 * ranks).astype(int)
size = min(n1, n2)
observed = doubled[:n1].sum() if n1 <= n2 else doubled[n1:].sum()
top = np.sort(doubled)[total - size:].sum()
if (size + 1) * (top + 1) * total > 50_000_000:
    exact_p = np.nan
    print("\nexact P: not computed, past 50 million cells of work (the app shows none either)")
else:
    counts = np.zeros((size + 1, top + 1))
    counts[0, 0] = 1.0
    for rank in doubled:
        counts[1:, rank:] += counts[:-1, : top + 1 - rank].copy()
    null = counts[size] / counts[size].sum()
    exact_p = min(1.0, 2 * null[: observed + 1].sum(), 2 * null[observed:].sum())
    print(f"\nexact P (two-sided) = {exact_p:.6g}")

## Effect sizes

In [ ]:
# r = z / sqrt(N), with the tie-corrected z of the normal approximation (no continuity
# correction) signed so that r is positive when the first group sits higher. The
# rank-biserial correlation (Kerby 2014) is 1 - 2 U1 / (n1 n2): the share of pairs in which
# the first group's value is larger, less the share in which it is smaller. Both are read on
# Cohen's (1988) scale for a correlation: 0.1 small, 0.3 medium, 0.5 large.
r = (u2 - expected) / np.sqrt(variance) / np.sqrt(total)
rank_biserial = 1 - 2 * u1 / (n1 * n2)

# The Hodges-Lehmann shift is the median of every difference a_i - b_j. Its interval is the
# one R's wilcox.test(a, b, conf.int = TRUE) gives (R 4.6). With both groups under 50 it is
# exact: with no ties, the q-th smallest and q-th largest difference, q the 2.5% point of
# U; with ties, the differences at which the exact tail probability of the rank sum, given
# the ties, first passes 2.5% (it then covers at least 95%). From 50 a group on it is where
# the tie-corrected, continuity-corrected z crosses +-1.96. The statistic only changes at a
# difference, so each limit is a difference. The helper beside this script does the
# counting. The app holds every difference in memory and stops past 10 million; this
# script does not.
diffs = np.subtract.outer(a, b).ravel()
shift = np.median(diffs)
# Groups too small for any 95% interval: R gives -Inf to Inf, and here none is shown.
limits = hodges_lehmann_interval(a, b, 0.95)
effect_sizes = pd.DataFrame(
    {"value": [r, rank_biserial, shift],
     "95% lower": [np.nan, np.nan, limits[0]], "95% upper": [np.nan, np.nan, limits[1]]},
    index=["r = z / sqrt(N)", "rank-biserial correlation", "Hodges-Lehmann shift"])
print()
print(effect_sizes)

## What the result says

The Mann-Whitney U test sorts all the tickets from fastest to slowest and compares the ranks, the positions in that order. The mean rank is 11.5 for the chatbot and 19 for the human agent, so chatbot tickets tend to be solved sooner. The statistic U is 36.

Some times are tied, so the printed table does not apply. The exact test gives a two-sided P value of 0.019, and the normal approximation with continuity correction gives z = 2.29 and P = 0.022. The P value is the chance of a gap this large if neither channel were faster. The rank-biserial correlation, an effect size, is -0.56, which counts as large. The Hodges-Lehmann shift, a typical difference between the channels, is -7 minutes, with a 95% confidence interval (the range of plausible values) from -10 to -4.

A chatbot ticket is usually solved first, typically about 7 minutes sooner. Ranks hide how slow the few stuck chatbot tickets are.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.